In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm import tqdm

save_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/multigraph_manual")
metadata_dir = save_dir / "metadata"

gauges = gpd.read_parquet(metadata_dir / "gauges.parquet").set_index('site_id')

subs_df = pd.read_parquet(metadata_dir / 'subbasins.parquet', columns=['site_id', 'is_gauge', 'uparea_km2', 'outlet_id'])
subs_df.index = subs_df.index.astype(str)

basin_subbasin_dict = subs_df.groupby('outlet_id').apply(lambda g: g.index.tolist(), include_groups=False).to_dict()

In [2]:
from data import ZarrBasinStore
store_path = Path("/scratch4/workspace/tlanghorst_umass_edu-zbs_vD")
store = ZarrBasinStore(store_path)

In [ ]:
store.compute_and_store_stats()

Dask client started. Dashboard at: http://127.0.0.1:8787/status


Basins:  32%|███▏      | 464/1473 [00:23<00:50, 19.98it/s]

In [3]:
from data import BasinDataLake

datalake = BasinDataLake('/nas/cee-water/cjgleason/ted/swot-ml/data/multigraph_manual/datalakes/training')
# datalake.optimize(0)

In [5]:
import xarray as xr

sub_counts = subs_df['outlet_id'].value_counts().to_dict()
subs_df = subs_df.sort_values(by='outlet_id', key = lambda x: x.map(sub_counts), ascending=False)

BATCH_SIZE = 1000
batches = []
for basin_id, basin_group in subs_df.groupby('outlet_id'):
    basin_subs = basin_subbasin_dict[basin_id]x
    n_subs = len(basin_subs)

    for i in range(0, n_subs, BATCH_SIZE):
        batch_subs = basin_subs[i : i + BATCH_SIZE]
        batches.append((basin_id, batch_subs))

def get_batch_era5_df(basin_id, batch_subs):
    root = save_dir / f"datalakes/training/dynamic_data/era5/basin={basin_id}"

    dfs = []
    for d in root.iterdir():
        tmp_df = pd.concat([pd.read_parquet(f) for f in d.glob("*.parquet")])
        tmp_df = tmp_df.set_index(['subbasin','date'])
        tmp_df = tmp_df.loc[pd.IndexSlice[batch_subs, :], :]
        dfs.append(tmp_df)
    
    df = pd.concat(dfs)
    df = df.loc[~df.index.duplicated(keep='first'), :]
    return df

processed_basins = set()
failed_basins = {}
for basin_id, batch_subs in tqdm(batches):
    basin_path = store.store_path / basin_id 
    if (basin_path / 'ro_mean').is_dir():
        ds = xr.open_zarr(basin_path)
        nan_count = ds['ro_mean'].isnull().sum().compute().item()
        if nan_count == 0:
            continue
    
    basin_subs = basin_subbasin_dict[basin_id]
    batch_df_list = []

    try:
        batch_df = get_batch_era5_df(basin_id, batch_subs)
        init_vars = basin_id not in processed_basins
        store.write_batch_data(basin_id, batch_df, basin_subs, batch_subs, init_vars)
        processed_basins.union({basin_id})
    except Exception as e: 
        failed_basins[basin_id] = e
        raise e

100%|██████████| 1508/1508 [44:29<00:00,  1.77s/it]  


In [ ]:
266

In [6]:
batch_df

ssrd_mean     strd_mean  \
subbasin       date                                                    
56155034       2015-01-01 00:00:00+00:00  3.176756e+07  2.818138e+07   
               2015-01-02 00:00:00+00:00  3.223902e+07  2.863344e+07   
               2015-01-03 00:00:00+00:00  2.381179e+07  2.815478e+07   
               2015-01-04 00:00:00+00:00  3.275706e+07  2.696731e+07   
               2015-01-05 00:00:00+00:00  3.288375e+07  2.992960e+07   
...                                                ...           ...   
ABOM-102984010 2025-12-27 00:00:00+00:00  1.834709e+07  2.877304e+07   
               2025-12-28 00:00:00+00:00  3.235656e+07  2.654268e+07   
               2025-12-29 00:00:00+00:00  3.310289e+07  2.744607e+07   
               2025-12-30 00:00:00+00:00  3.296509e+07  2.709251e+07   
               2025-12-31 00:00:00+00:00  3.309605e+07  2.732326e+07   

                                               ro_mean      sro_mean  \
subbasin       date                                                    
56155034       2015-01-01 00:00:00+00:00  1.503372e-08  0.000000e+00   
               2015-01-02 00:00:00+00:00  1.722280e-08  0.000000e+00   
               2015-01-03 00:00:00+00:00  2.980232e-08  0.000000e+00   
               2015-01-04 00:00:00+00:00  2.980232e-08  0.000000e+00   
               2015-01-05 00:00:00+00:00  2.246098e-08  0.000000e+00   
...                                                ...           ...   
ABOM-102984010 2025-12-27 00:00:00+00:00  2.607327e-06  4.193518e-08   
               2025-12-28 00:00:00+00:00  2.535080e-06  0.000000e+00   
               2025-12-29 00:00:00+00:00  2.517255e-06  0.000000e+00   
               2025-12-30 00:00:00+00:00  2.469628e-06  0.000000e+00   
               2025-12-31 00:00:00+00:00  2.456665e-06  2.285194e-09   

                                            e_mean       tp_mean  \
subbasin       date                                                
56155034       2015-01-01 00:00:00+00:00 -0.000341  1.290134e-06   
               2015-01-02 00:00:00+00:00 -0.000444  2.145250e-06   
               2015-01-03 00:00:00+00:00 -0.000251  1.299381e-06   
               2015-01-04 00:00:00+00:00 -0.000308  2.990087e-06   
               2015-01-05 00:00:00+00:00 -0.000424  8.574305e-07   
...                                            ...           ...   
ABOM-102984010 2025-12-27 00:00:00+00:00 -0.000977  2.342487e-05   
               2025-12-28 00:00:00+00:00 -0.001240  2.131183e-06   
               2025-12-29 00:00:00+00:00 -0.001236  1.284480e-06   
               2025-12-30 00:00:00+00:00 -0.001250  8.523463e-07   
               2025-12-31 00:00:00+00:00 -0.001217  2.137012e-06   

                                               sp_mean    d2m_mean  \
subbasin       date                                                  
56155034       2015-01-01 00:00:00+00:00  97239.430646  281.680918   
               2015-01-02 00:00:00+00:00  97409.321352  282.850554   
               2015-01-03 00:00:00+00:00  97787.337662  278.965451   
               2015-01-04 00:00:00+00:00  97599.654488  282.527995   
               2015-01-05 00:00:00+00:00  97239.071113  282.829329   
...                                                ...         ...   
ABOM-102984010 2025-12-27 00:00:00+00:00  98453.178023  278.708177   
               2025-12-28 00:00:00+00:00  98552.394148  281.244677   
               2025-12-29 00:00:00+00:00  98813.668509  280.405255   
               2025-12-30 00:00:00+00:00  98675.470554  280.409691   
               2025-12-31 00:00:00+00:00  98134.370378  279.553260   

                                            t2m_mean  fal_mean      ssrd_var  \
subbasin       date                                                            
56155034       2015-01-01 00:00:00+00:00  295.512250  0.210996  2.469542e+10   
               2015-01-02 00:00:00+00:00  292.707864  0.211212  1.562200e+09   
               2015-01-03 00:00:00+00:00  292.915523  0

In [52]:

tmp = pd.read_parquet('/nas/cee-water/cjgleason/ted/swot-ml/data/multigraph_manual/datalakes/training/dynamic_data/era5/basin=USGS-14246900/year=1983/part-00002-9369e52c-7378-429b-88c6-ddb97b5dbc96-c000.zstd.parquet')

tmp

,date,ssrd_mean,strd_mean,ro_mean,sro_mean,e_mean,tp_mean,sp_mean,d2m_mean,t2m_mean,...,strd_var,ro_var,sro_var,e_var,tp_var,sp_var,d2m_var,t2m_var,fal_var,subbasin
0,1983-12-31 00:00:00+00:00,5.164583e+06,2.400597e+07,0.000008,8.082325e-06,-0.000167,0.005205,100143.602950,265.285063,266.475176,...,1.957817e+10,4.397316e-12,2.883222e-12,1.434406e-09,2.415832e-07,6.508864e+05,0.346675,0.391446,1.359393e-03,78017400
1,1983-12-31 00:00:00+00:00,5.520677e+06,2.509972e+07,0.000029,1.563233e-05,-0.000101,0.008436,96955.163018,270.627767,272.537437,...,3.860369e+10,3.617793e-10,5.334791e-11,7.052063e-10,1.269360e-07,5.611102e+05,0.181795,0.195899,5.905695e-06,78017445
2,1983-12-31 00:00:00+00:00,5.479941e+06,2.473411e+07,0.000022,1.469519e-05,-0.000143,0.007062,97254.040425,269.444913,270.921102,...,3.067791e+10,1.255174e-10,9.433805e-12,1.376925e-09,2.390943e-07,6.646891e+05,0.422507,0.666880,1.465186e-04,78017945
3,1983-12-31 00:00:00+00:00,4.964483e+06,2.570039e+07,0.000139,8.482945e-05,-0.000047,0.009384,96602.703082,271.919168,273.313719,...,8.649024e+09,3.428317e-10,8.126469e-11,1.427461e-10,1.661545e-07,8.817612e+05,0.010186,0.019848,1.477066e-07,USGS-14015000
4,1983-12-31 00:00:00+00:00,5.628195e+06,2.469006e+07,0.000045,1.092153e-05,-0.000096,0.007688,97485.255375,269.550946,271.172536,...,6.287276e+10,2.790962e-10,3.328955e-11,7.368626e-10,5.766100e-08,5.530347e+05,0.383774,0.731503,5.345863e-05,78017442
5,1983-12-31 00:00:00+00:00,5.105030e+06,2.545666e+07,0.000085,5.891747e-05,-0.000087,0.008340,98382.766058,271.537618,272.834428,...,9.881400e+09,5.888247e-10,1.356858e-10,4.031517e-10,1.691273e-07,1.578291e+05,0.044417,0.061047,4.463079e-06,USGS-14015350
6,1983-12-31 00:00:00+00:00,5.113936e+06,2.405339e+07,0.000082,1.026970e-06,-0.000096,0.003950,96422.876651,269.800787,270.530155,...,8.108241e+10,3.011797e-09,1.287740e-12,1.229453e-09,7.306089e-08,5.829162e+06,1.060154,0.863114,9.334200e-04,78017618
7,1983-12-31 00:00:00+00:00,5.195957e+06,2.373036e+07,0.000008,6.776523e-06,-0.000158,0.003316,98365.302834,265.505926,266.545378,...,8.919912e+09,3.647940e-12,1.092894e-12,9.681225e-10,3.503883e-08,3.332937e+05,2.699607,2.249719,7.626489e-04,78020026
8,1983-12-31 00:00:00+00:00,5.110880e+06,2.396591e+07,0.000006,6.238745e-06,-0.000304,0.003482,99374.581412,267.747477,268.635864,...,8.721581e+09,1.496859e-12,1.519885e-12,4.844037e-08,4.017751e-08,1.629034e+06,2.104439,2.309446,4.213032e-03,78019699
9,1983-12-31 00:00:00+00:00,5.172596e+06,2.375831e+07,0.000005,4.504469e-06,-0.000134,0.004162,99856.591976,264.320887,265.353746,...,1.619937e+10,3.086450e-12,3.061086e-12,1.647948e-10,6.775354e-08,1.744837e+06,0.401196,0.356935,1.058512e-04,78017403


In [53]:
len(tmp['subbasin'].unique())

21

In [45]:
basin_id

'USGS-14246900'

In [44]:
len(batch_subs)

1000

In [41]:
len(tmp_df.index.get_level_values('subbasin').unique())

21

In [42]:
len(dfs[0].index.get_level_values('subbasin').unique())

1000

In [35]:
root = save_dir / f"datalakes/training/dynamic_data/era5/basin={basin_id}"
    
dfs = []
for f in root.glob("*/*.parquet"):
    tmp_df = pd.read_parquet(f).set_index(['subbasin','date'])
    # tmp_df = tmp_df.loc[pd.IndexSlice[batch_subs, :], :]
    dfs.append(tmp_df)
df = pd.concat(dfs)

df

KeyError: '78027647'

In [32]:
failed_basins

{'USGS-14246900': "'78027647'"}

In [6]:
basin_subs = basin_subbasin_dict[basin_id]
batch_df_list = []


In [11]:
import xarray as xr
basin_path = store.store_path / basin_id
ds = xr.open_zarr(basin_path)

ds

<xarray.Dataset> Size: 61MB
Dimensions:         (date: 16437, subbasin: 20)
Coordinates:
  * date            (date) datetime64[ns] 131kB 1980-01-01 ... 2024-12-31
  * subbasin        (subbasin) <U14 1kB '56155034' ... 'ABOM-102984010'
Data variables: (12/23)
    d2m_mean        (date, subbasin) float64 3MB dask.array<chunksize=(365, 20), meta=np.ndarray>
    d2m_var         (date, subbasin) float64 3MB dask.array<chunksize=(365, 20), meta=np.ndarray>
    discharge       (date, subbasin) float64 3MB dask.array<chunksize=(365, 20), meta=np.ndarray>
    e_mean          (date, subbasin) float64 3MB dask.array<chunksize=(365, 20), meta=np.ndarray>
    e_var           (date, subbasin) float64 3MB dask.array<chunksize=(365, 20), meta=np.ndarray>
    fal_mean        (date, subbasin) float64 3MB dask.array<chunksize=(365, 20), meta=np.ndarray>
    ...              ...
    strd_var        (date, subbasin) float64 3MB dask.array<chunksize=(365, 20), meta=np.ndarray>
    t2m_mean        (date, subbasin) float64 3MB dask.array<chunksize=(365, 20), meta=np.ndarray>
    t2m_var         (date, subbasin) float64 3MB dask.array<chunksize=(365, 20), meta=np.ndarray>
    tp_mean         (date, subbasin) float64 3MB dask.array<chunksize=(365, 20), meta=np.ndarray>
    tp_var          (date, subbasin) float64 3MB dask.array<chunksize=(365, 20), meta=np.ndarray>
    unit_discharge  (date, subbasin) float64 3MB dask.array<chunksize=(365, 20), meta=np.ndarray>
Attributes:
    normalization_stats:  {'d2m_mean': {'count': 328740, 'sum': 92463169.5251...

In [24]:
ds['ro_mean'].isnull().sum().compute().item()